In [ ]:
# streamlit run your_app.py --server.address localhost


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 🔐 Load environment variables from .env

openai_api_key = os.getenv("OPENAI_API_KEY")

# print(openai_api_key)

sk-xxxxxxx


Below is one possible approach to building a modular Streamlit app that processes PDF files with an LLM, tags the extracted sentences, displays relevant information, and offers a download option.

---

### Directory Structure



In [ ]:
streamlit/
├── app.py                # Main Streamlit app
├── my_modules/
│   ├── pdf_utils.py      # PDF processing functions
│   ├── llm_utils.py      # LLM interaction functions
│   └── tag_utils.py      # Tag-related utilities
└── tag_definitions.csv   # CSV file for tag descriptions



---

#### app.py


In [ ]:
'''python
# filepath: /Users/rahulaggarwal/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/mr-rahul-aggarwal-git-hub/online-courses/streamlit/app.py
import streamlit as st
import pandas as pd
import os

from my_modules.pdf_utils import parse_pdf
from my_modules.llm_utils import extract_sentences, identify_tags
from my_modules.tag_utils import fetch_tag_descriptions

st.title("LLM-Powered Document Analyzer")
st.markdown("<hr style='border:1px solid blue'>", unsafe_allow_html=True)

def main():
    uploaded_file = st.file_uploader("Upload a PDF file", type=["pdf"])
    if uploaded_file is not None:
        with open("temp.pdf", "wb") as f:
            f.write(uploaded_file.getbuffer())

        # Parse the PDF into pages
        pages_content = parse_pdf("temp.pdf")

        # Extract relevant sentences using LLM
        relevant_sentences = extract_sentences(pages_content)

        # Identify tags for each sentence
        tagged_sentences = identify_tags(relevant_sentences)

        # Load external tag descriptions
        tag_info = fetch_tag_descriptions()

        # Display results
        st.subheader("Relevant Pages and Sentences")
        for item in tagged_sentences:
            page_number = item["page"]
            sentence = item["sentence"]
            tags = item["tags"]

            st.markdown(f"**Page {page_number}:** {sentence}")
            for tag in tags:
                # Tag button with tooltip from CSV definitions
                description = tag_info.get(tag, "No description found.")
                st.button(tag, help=description)
            st.write("---")

        # Download option
        if st.button("Download Results"):
            df = pd.DataFrame(tagged_sentences)
            filename = "extracted_data.csv"
            df.to_csv(filename, index=False)
            with open(filename, "rb") as file:
                btn = st.download_button(
                    label="Download CSV",
                    data=file,
                    file_name=filename,
                    mime="text/csv"
                )

if __name__ == "__main__":
    main()
```



---

#### `pdf_utils.py`


In [ ]:
```python
# filepath: /Users/rahulaggarwal/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/mr-rahul-aggarwal-git-hub/online-courses/streamlit/my_modules/pdf_utils.py
import PyPDF2

def parse_pdf(pdf_path):
    """
    Extracts text from each page of the PDF and returns a list of page contents.
    """
    pages_content = []
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            pages_content.append({"page": i+1, "content": text})
    return pages_content
```



---

#### `llm_utils.py`


In [ ]:
```python
# filepath: /Users/rahulaggarwal/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/mr-rahul-aggarwal-git-hub/online-courses/streamlit/my_modules/llm_utils.py
# Placeholder for AzureOpenAI (or any future LLM)

def extract_sentences(pages_content):
    """
    Calls an LLM to find relevant sentences from the pages_content.
    Returns a list of dicts with 'page' and 'sentence'.
    """
    # Example stub logic:
    relevant_sentences = []
    for pg_obj in pages_content:
        page_number = pg_obj["page"]
        # Placeholder: In real usage, call LLM to extract key sentences
        # Here, assume we just take the first line
        lines = pg_obj["content"].split("\n")
        if lines:
            relevant_sentences.append({
                "page": page_number,
                "sentence": lines[0]
            })
    return relevant_sentences

def identify_tags(relevant_sentences):
    """
    Calls an LLM to generate tags for each sentence.
    Returns a list of dicts with 'page', 'sentence', and 'tags'.
    """
    # Example stub logic:
    # In real usage, pass sentence to GPT or AzureOpenAI for tagging
    final_data = []
    for sent_obj in relevant_sentences:
        sentence = sent_obj["sentence"]
        # Placeholder tags
        tags = ["TagA", "TagB"]
        final_data.append({
            "page": sent_obj["page"],
            "sentence": sentence,
            "tags": tags
        })
    return final_data
```



---

#### `tag_utils.py`


In [ ]:
```python
# filepath: /Users/rahulaggarwal/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/mr-rahul-aggarwal-git-hub/online-courses/streamlit/my_modules/tag_utils.py
import pandas as pd

def fetch_tag_descriptions():
    """
    Reads a CSV file to retrieve tag descriptions.
    Returns a dict: { 'TagName': 'Tag Description', ... }
    """
    # Example CSV layout:
    # tag_name,description
    # TagA,Description for Tag A
    # TagB,Description for Tag B
    
    data = pd.read_csv("tag_definitions.csv")
    tag_dict = dict(zip(data["tag_name"], data["description"]))
    return tag_dict
```



---

### Notes
1. **LLM Interaction**: Replace the stub logic in `llm_utils.py` with actual Azure OpenAI or GPT API calls.  
2. **Future File Types**: Extend `pdf_utils.py` or create additional modules to handle various file formats.  
3. **Modularity**: You can move or rename modules without breaking other parts of the system.  
4. **Tag Descriptions**: Maintain a CSV or a database table for easy updates.